# Şəkil Generasiyası — Hugging Face (Diffusers)

Bu notebook `diffusers` kitabxanası ilə mətndən şəkil (text-to-image) generasiyasını göstərir.

## Diffusion modelləri necə işləyir?

Diffusion modeli təsadüfi "səs-küydən" (noise) başlayır və addım-addım onu təmizləyərək (denoising) mənalı şəkil halına gətirir. Hər addımda model mətn təsvirinə (prompt) əsaslanaraq şəkli bir az daha aydınlaşdırır. `num_inference_steps` — bu addımların sayıdır (çox olsa keyfiyyət artır, amma vaxt da artır).

> **Colab GPU:** İşə başlamazdan əvvəl: `Runtime → Change runtime type → GPU (T4)` seç. GPU olmadan bu modellər ya işləməyəcək, ya da çox yavaş olacaq.

In [ ]:
!pip install -q diffusers transformers accelerate

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

### Modeli yükləmək
Stable Diffusion 1.5 — nisbətən yüngül, free T4-də rahat işləyir.

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)
pipe = pipe.to("cuda")

### Şəkil generasiyası

In [ ]:
prompt = "a witch coding at night, digital art, cyberpunk style"
negative_prompt = "blurry, low quality, distorted"

image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5
).images[0]

image.save("output.png")
image

### Parametrlər nə deməkdir?

- `negative_prompt` — şəkildə **görmək istəmədiyin** şeylər
- `guidance_scale` — modelin prompt-a nə qədər "sadiq" qalacağı (7–9 arası tarazlıqdır, çox yüksək olsa şəkil süni görünə bilər)
- `num_inference_steps` — 20–50 arası kifayətdir, 50-dən sonra fərq az hiss olunur

### Bir neçə prompt sınayaq

In [ ]:
prompts = [
    "a futuristic Baku skyline at night, neon lights",
    "a cozy library with warm lighting, watercolor style",
    "a robot reading a book in a garden, studio ghibli style",
]

images = []
for p in prompts:
    img = pipe(p, num_inference_steps=30).images[0]
    images.append(img)
    img.save(f"{p[:20].replace(' ', '_')}.png")

images[0]

## Növbəti addımlar

- Daha yüksək keyfiyyət üçün `stabilityai/stable-diffusion-xl-base-1.0` (SDXL) sına — daha çox VRAM istəyir
- `pipe.enable_model_cpu_offload()` yaddaş azdırsa köməkdir
- LoRA fine-tuning ilə öz stilini öyrətmək mümkündür
- `img2img` pipeline ilə mövcud şəkli dəyişdirmək olar